# Attention Is All You Need — Visual Notebook

This notebook turns the Transformer paper into a visual, explainable walkthrough.

Source: *Attention Is All You Need* (Vaswani et al., 2017)

## 1. What the paper proposes

The paper introduces the **Transformer**, a sequence-to-sequence model that removes recurrence and convolution entirely and relies only on attention.

### Core ideas
- **Self-attention** connects every token to every other token.
- **Multi-head attention** lets the model look at different relationships in parallel.
- **Positional encoding** injects order information because the model has no recurrence.
- The result is **faster training** and strong translation quality.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import pandas as pd

plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

## 2. Transformer architecture at a glance

The encoder and decoder are both stacks of repeated layers.

- Encoder layer: **self-attention → feed-forward**
- Decoder layer: **masked self-attention → encoder-decoder attention → feed-forward**
- Residual connections and layer normalization appear around each sub-layer

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis("off")

def box(x, y, w, h, text, fc="#f4f4f4", ec="black", fontsize=11, weight="normal"):
    patch = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.08",
                           linewidth=1.5, edgecolor=ec, facecolor=fc)
    ax.add_patch(patch)
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=fontsize, weight=weight)

def arrow(x1, y1, x2, y2):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="->", mutation_scale=14, linewidth=1.4))

ax.text(3.5, 9.3, "Encoder", fontsize=18, weight="bold", ha="center")
ax.text(10.5, 9.3, "Decoder", fontsize=18, weight="bold", ha="center")

# Encoder stack
box(1.2, 6.8, 4.6, 1.0, "Input Embedding + Positional Encoding", fc="#f8d9d9", fontsize=12, weight="bold")
box(1.5, 5.2, 4.0, 1.0, "Multi-Head Self-Attention", fc="#ffe2b8", fontsize=12)
box(1.5, 3.5, 4.0, 1.0, "Feed Forward", fc="#d7ecff", fontsize=12)
box(1.2, 1.7, 4.6, 1.0, "Encoder Output (N × layers)", fc="#eeeeee", fontsize=12, weight="bold")

arrow(3.5, 6.8, 3.5, 6.2)
arrow(3.5, 5.2, 3.5, 4.5)
arrow(3.5, 3.5, 3.5, 2.7)

# Decoder stack
box(7.8, 6.8, 4.6, 1.0, "Output Embedding + Positional Encoding", fc="#f8d9d9", fontsize=12, weight="bold")
box(8.1, 5.2, 4.0, 1.0, "Masked Multi-Head Self-Attention", fc="#ffe2b8", fontsize=11)
box(8.1, 3.7, 4.0, 1.0, "Encoder-Decoder Attention", fc="#ffe2b8", fontsize=11)
box(8.1, 2.2, 4.0, 1.0, "Feed Forward", fc="#d7ecff", fontsize=12)
box(7.8, 0.5, 4.6, 1.0, "Decoder Output", fc="#eeeeee", fontsize=12, weight="bold")

arrow(10.1, 6.8, 10.1, 6.2)
arrow(10.1, 5.2, 10.1, 4.7)
arrow(10.1, 3.7, 10.1, 3.2)
arrow(10.1, 2.2, 10.1, 1.5)

# Cross-attention arrow from encoder output to decoder attention
arrow(5.8, 2.2, 8.1, 4.2)
ax.text(6.7, 3.2, "memory from encoder", fontsize=10, rotation=18, ha="center")

ax.text(3.5, 0.9, "Residual connections + layer normalization around each sub-layer", ha="center", fontsize=11)
ax.text(10.5, 9.8, "N = 6 repeated layers in the paper's base model", ha="center", fontsize=11)

plt.tight_layout()
plt.show()

## 3. Scaled dot-product attention

The paper defines attention as:

\[
\text{Attention}(Q, K, V) = \mathrm{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
\]

The scaling by \(\sqrt{d_k}\) keeps the dot products from becoming too large.

In [ ]:
def scaled_dot_product_attention(q, k, v):
    dk = q.shape[-1]
    scores = q @ k.T / np.sqrt(dk)
    weights = np.exp(scores - scores.max(axis=-1, keepdims=True))
    weights = weights / weights.sum(axis=-1, keepdims=True)
    output = weights @ v
    return scores, weights, output

# Toy example
np.random.seed(7)
q = np.random.randn(5, 4)
k = np.random.randn(5, 4)
v = np.random.randn(5, 3)

scores, weights, output = scaled_dot_product_attention(q, k, v)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
im0 = axes[0].imshow(weights, aspect="auto")
axes[0].set_title("Attention Weights")
axes[0].set_xlabel("Key positions")
axes[0].set_ylabel("Query positions")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(output, aspect="auto")
axes[1].set_title("Weighted Output")
axes[1].set_xlabel("Output features")
axes[1].set_ylabel("Query positions")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## 4. Why multi-head attention matters

Instead of one attention operation, the Transformer uses several heads in parallel.

This helps the model:
- learn different relationships simultaneously,
- attend to different positions and representation subspaces,
- avoid the averaging effect of a single head.

In [ ]:
heads = pd.DataFrame({
    "Setting": ["1 head", "4 heads", "8 heads", "16 heads"],
    "Model idea": ["single focus", "fewer parallel views", "paper's base model", "many narrow heads"],
    "Relative capacity": [0.55, 0.78, 1.00, 0.97],
})

display(heads)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(heads["Setting"], heads["Relative capacity"])
ax.set_title("Conceptual Multi-Head Attention Comparison")
ax.set_ylabel("Relative capacity")
ax.set_ylim(0, 1.2)
plt.tight_layout()
plt.show()

## 5. Positional encoding

Because there is no recurrence or convolution, the model needs explicit position information.

The paper uses sine and cosine waves with different frequencies:

\[
PE_{pos,2i} = \sin\left(pos / 10000^{2i/d_{model}}\right)
\]
\[
PE_{pos,2i+1} = \cos\left(pos / 10000^{2i/d_{model}}\right)
\]

This gives every position a unique pattern and helps the model learn relative order.

In [ ]:
def positional_encoding(max_len=60, d_model=16):
    pe = np.zeros((max_len, d_model))
    position = np.arange(max_len)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    return pe

pe = positional_encoding(60, 16)

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(pe.T, aspect="auto", interpolation="nearest")
ax.set_title("Sinusoidal Positional Encoding (16 dimensions, 60 positions)")
ax.set_xlabel("Position")
ax.set_ylabel("Dimension")
fig.colorbar(im, ax=ax, fraction=0.025, pad=0.04)
plt.tight_layout()
plt.show()

## 6. What the paper reports in results

The paper shows strong translation quality on WMT 2014:
- **English → German:** 28.4 BLEU for the big model
- **English → French:** 41.0 BLEU for the big model

The key message is not only accuracy, but also that the model trains faster and parallelizes better than recurrent approaches.

In [ ]:
results = pd.DataFrame({
    "Model": ["Transformer (base)", "Transformer (big)"],
    "EN-DE BLEU": [27.3, 28.4],
    "EN-FR BLEU": [38.1, 41.0],
    "Training steps": ["100K", "300K"],
})

display(results)

fig, ax = plt.subplots(figsize=(8.5, 4.8))
x = np.arange(len(results))
width = 0.28
ax.bar(x - width/2, results["EN-DE BLEU"], width, label="EN-DE")
ax.bar(x + width/2, results["EN-FR BLEU"], width, label="EN-FR")
ax.set_xticks(x)
ax.set_xticklabels(results["Model"])
ax.set_ylabel("BLEU")
ax.set_title("Reported BLEU Scores")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Quick summary

The Transformer changed sequence modeling by showing that attention alone could outperform recurrent and convolutional sequence transduction models.

### The three most important takeaways
1. **Self-attention replaces recurrence**
2. **Multi-head attention improves expressiveness**
3. **Positional encoding restores order information**

## 8. Suggested exercises

Try extending this notebook by:
- drawing a real attention matrix for a sentence,
- visualizing masked attention for the decoder,
- comparing the Transformer with an RNN in a timing benchmark,
- annotating the figure from page 3 of the paper with your own labels.